# Notebook 3 — Import the Knowledge Graph into Neo4j

This is the **third step** of the GraphRAG pipeline. It reads the JSON entity/relationship files produced in Notebook 2 and loads them into **Neo4j** as a property graph.

## What this notebook does
1. Connects to Neo4j and creates a uniqueness constraint on `Entity.key`
2. Parses each `.txt` (JSON) file in `data/wikipedia_entities/`
3. **Upserts entities** — each unique entity becomes an `(:Entity)` node with `key`, `name`, `type`, and `description` properties. Entities that appear across multiple articles are merged into a single node.
4. **Upserts relationships** — each triple becomes a typed directed edge, e.g. `(marie_curie)-[:COLLABORATED_WITH]->(pierre_curie)`

## Graph schema
```
(:Entity {key, name, type, description})
    -[:RELATIONSHIP_TYPE {evidence, confidence}]->
(:Entity {key, name, type, description})
```

Entity types include: `Person`, `Field`, `Discovery`, `Invention`, `Organization`, `Award`, `Place`, `Publication`, and more.

## Why MERGE instead of CREATE?
Using `MERGE` means re-running this notebook only updates existing nodes/relationships rather than creating duplicates. The `key` property is a lowercase slug derived from the entity name.

## Output
A populated Neo4j database ready for multi-hop graph queries.

## Next step
Run **Notebook 4** to create vector embeddings of the Wikipedia articles and store them in Qdrant.

## 1 · Imports

In [16]:
import json
import os
import re
import time
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

print(f"Importing libraries done!")

Importing libraries done!


## 2 · Configuration

In [17]:
SCRIPT_DIR = Path(".").resolve()
load_dotenv(dotenv_path=SCRIPT_DIR.parent / ".env")

NEO4J_URI      = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

ENTITIES_DIR = SCRIPT_DIR.parent / "data" / "wikipedia_entities"

print(f"NEO4J Configuration done!")
print(f"Entities directory : {ENTITIES_DIR}")
print(f"Neo4j URI          : {NEO4J_URI}")

NEO4J Configuration done!
Entities directory : C:\Repos\GraphRAG_Demo\src\data\wikipedia_entities
Neo4j URI          : bolt://localhost:7687


## 3 · Helper Functions

In [18]:
def make_key(name: str) -> str:
    """Create a stable, lowercase slug from an entity name for use as a unique key."""
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def parse_json_file(path: Path) -> dict | None:
    """Read a file that may contain raw JSON or JSON wrapped in a markdown code block."""
    raw = path.read_text(encoding="utf-8").strip()
    # Strip markdown code fences if present
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"  [warn] Could not parse {path.name}: {e}")
        return None

## 4 · Neo4j Operations

In [19]:
def create_constraints(driver):
    driver.execute_query(
        """
        CREATE CONSTRAINT entity_key_unique IF NOT EXISTS
        FOR (e:Entity)
        REQUIRE e.key IS UNIQUE
        """,
        database_=NEO4J_DATABASE,
    )


def upsert_entities(driver, entities: list[dict]) -> int:
    # Group by entity type so we can apply it as a dynamic label
    grouped: dict[str, list[dict]] = {}
    for e in entities:
        # Sanitize type into a valid Cypher label (letters, digits, underscores)
        label = re.sub(r"[^A-Za-z0-9]", "_", e.get("type", "Unknown")).strip("_") or "Unknown"
        grouped.setdefault(label, []).append(e)

    total = 0
    for label, group in grouped.items():
        rows = [
            {
                "key":         make_key(e["name"]),
                "name":        e.get("name", ""),
                "type":        e.get("type", ""),
                "description": e.get("description", ""),
            }
            for e in group
        ]
        # The entity type is added as a second label (e.g. :Entity:Person)
        # so Neo4j Browser assigns a distinct color per type automatically
        driver.execute_query(
            f"""
            UNWIND $rows AS row
            MERGE (e:Entity {{key: row.key}})
            SET e:{label},
                e.name        = row.name,
                e.type        = row.type,
                e.description = row.description
            """,
            rows=rows,
            database_=NEO4J_DATABASE,
        )
        total += len(rows)
    return total


def upsert_relationships(driver, relationships: list[dict]) -> int:
    # Group by relationship type (Cypher requires literal type names)
    grouped: dict[str, list[dict]] = {}
    for rel in relationships:
        rel_type = re.sub(r"[^A-Z0-9_]", "_", rel["relationship"].upper())
        grouped.setdefault(rel_type, []).append(rel)

    total = 0
    for rel_type, rels in grouped.items():
        rows = [
            {
                "source_key": make_key(r["source"]),
                "target_key": make_key(r["target"]),
                "evidence":   r.get("evidence", ""),
                "confidence": float(r.get("confidence", 1.0)),
            }
            for r in rels
        ]
        driver.execute_query(
            f"""
            UNWIND $rows AS row
            MATCH (source:Entity {{key: row.source_key}})
            MATCH (target:Entity {{key: row.target_key}})
            MERGE (source)-[r:{rel_type}]->(target)
            SET r.evidence   = row.evidence,
                r.confidence = row.confidence
            """,
            rows=rows,
            database_=NEO4J_DATABASE,
        )
        total += len(rows)
    return total


## 5 · Run Import

In [20]:
entity_files = sorted(ENTITIES_DIR.glob("*.txt"))
if not entity_files:
    print(f"No files found in {ENTITIES_DIR}")
else:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    try:
        driver.verify_connectivity()
        print("Connected to Neo4j")

        create_constraints(driver)

        total_entities      = 0
        total_relationships = 0

        for path in entity_files:
            print(f"\n--- Importing: {path.name} ---")
            data = parse_json_file(path)
            if data is None:
                continue

            entities      = data.get("entities", [])
            relationships = data.get("relationships", [])

            n = upsert_entities(driver, entities)
            r = upsert_relationships(driver, relationships)
            total_entities      += n
            total_relationships += r
            print(f"  {n} entities, {r} relationships")

        print(f"\nDone. Total: {total_entities} entities, {total_relationships} relationships imported.")

    finally:
        driver.close()

Connected to Neo4j

--- Importing: Ada_Lovelace.txt ---
  [warn] Could not parse Ada_Lovelace.txt: Unterminated string starting at: line 50 column 115 (char 6678)

--- Importing: Al-Khwarizmi.txt ---
  23 entities, 14 relationships

--- Importing: Alan_Turing.txt ---
  34 entities, 14 relationships

--- Importing: Albert_Einstein.txt ---
  [warn] Could not parse Albert_Einstein.txt: Unterminated string starting at: line 109 column 7 (char 6447)

--- Importing: Alexander_Fleming.txt ---
  30 entities, 17 relationships

--- Importing: Antoine_Lavoisier.txt ---
  27 entities, 14 relationships

--- Importing: Archimedes.txt ---
  30 entities, 13 relationships

--- Importing: Barbara_McClintock.txt ---
  29 entities, 14 relationships

--- Importing: Carl_Sagan.txt ---
  51 entities, 21 relationships

--- Importing: Caroline_Herschel.txt ---
  21 entities, 10 relationships

--- Importing: Cecilia_Payne-Gaposchkin.txt ---
  27 entities, 16 relationships

--- Importing: Charles_Darwin.txt ---
